<a href="https://colab.research.google.com/github/andimcfl/DESGARGAS_INEGI/blob/main/cve_pilar_con_alcaldia_poligonos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import pandas as pd
import requests
from io import BytesIO
import re
!pip install unidecode
import unidecode
import numpy as np
import geopandas as gpd
from urllib.parse import urlparse, parse_qs, unquote
from shapely.geometry import Point
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
import unicodedata
from sentence_transformers import SentenceTransformer
import hdbscan
from google.colab import drive
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity


Funciones introductorias

In [2]:
#Crear función clean_names
def clean_names(df):
    df = df.copy()
    new_cols = []

    for col in df.columns:
        col = col.strip()                   # quitar espacios al inicio/fin
        col = col.lower()                   # minúsculas
        col = unidecode.unidecode(col)      # quitar acentos
        col = re.sub(r"[^\w\s]", "", col)   # quitar caracteres raros
        col = re.sub(r"\s+", "_", col)      # espacios → _
        new_cols.append(col)

    df.columns = new_cols
    return df

In [3]:
#Función para expandir la url de maps y extraer las coordenadas latitud y longitud
def limpiar_url_google(url):
    try:
        if "google.com/sorry/index" in url:
            parsed = urlparse(url)
            qs = parse_qs(parsed.query)

            if "continue" in qs:
                return unquote(qs["continue"][0])

        return url

    except:
        return url

def expandir_url(url):
    try:
        r = requests.get(url, allow_redirects=True, timeout=15)
        url_final = r.url
        return limpiar_url_google(url_final)

    except:
        return None

def extraer_coordenadas(url):
    try:
        if not url:
            return None, None

        match = re.search(r'@(-?\d+\.\d+),(-?\d+\.\d+)', url)

        if match:
            lat, lon = match.groups()
            return float(lat), float(lon)

        return None, None

    except:
        return None, None


In [4]:
#Función para limpiar los nombres de los pilares
def limpiar_texto(texto):
    if pd.isna(texto):
        return None

    # pasar a minúsculas
    texto = texto.lower()

    # quitar acentos
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ascii','ignore').decode('utf-8')

    # quitar comillas y símbolos, dejar solo letras, números y espacios
    texto = re.sub(r'[^a-z0-9\s]', '', texto)

    # eliminar palabra "pilares"
    texto = re.sub(r'\bpilares\b', '', texto)

    # quitar espacios dobles
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto


### Puntos pilares extraídos de su página

In [5]:
file_id = "1hCY34KrXt4j20l1KIB29uLFhEZgTWbHtTvYn2x8Pq8A"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"

res = requests.get(url)
pilares_cdmx = pd.read_excel(BytesIO(res.content))
pilares_cdmx.head()


,nombre_pilar,direccion,latitud,longitud,horario
0,100 METROS,"Av Cien Metros y Anillo Periferico S/N , Santi...",19526523.0,-99158897.0,De 09:00 a 19:00
1,16 DE septiembre,Cruce de Anillo Periferico y Av. 16 de Septiem...,19280984.0,-9913733.0,De 09:00 a 19:00
2,20 DE noviembre,"Avenida Gran Canal s/n 1 , 5to TRAMO 20 DE NOV...",19447458.0,-99103176.0,De 09:00 a 19:00
3,8 DE agosto,"Río Becerra s/n , 8 de agosto",19394845.0,-99187943.0,De 09:00 a 19:00
4,ACAHUALTEPEC,"Calle Colorín s/n, Colonia 1 , Santiago Acahua...",19346043.0,-99008134.0,De 09:00 a 19:00


In [6]:
pilares_cdmx["latitud"] = (
    pilares_cdmx["latitud"]
    .astype(str)
    .str.replace(r"[^0-9]", "", regex=True)   # quitar basura
    .apply(lambda x: float(x[:2] + "." + x[2:]) if len(x) > 2 else None)
)
pilares_cdmx["longitud"] = (
    pilares_cdmx["longitud"]
    .astype(str)
    .str.replace(r"[^0-9]", "", regex=True)   # quitar basura
    .apply(lambda x: float("-" + x[:2] + "." + x[2:]) if len(x) > 2 else None)
)
pilares_cdmx

,nombre_pilar,direccion,latitud,longitud,horario
0,100 METROS,"Av Cien Metros y Anillo Periferico S/N , Santi...",19.526523,-99.158897,De 09:00 a 19:00
1,16 DE septiembre,Cruce de Anillo Periferico y Av. 16 de Septiem...,19.280984,-99.137330,De 09:00 a 19:00
2,20 DE noviembre,"Avenida Gran Canal s/n 1 , 5to TRAMO 20 DE NOV...",19.447458,-99.103176,De 09:00 a 19:00
3,8 DE agosto,"Río Becerra s/n , 8 de agosto",19.394845,-99.187943,De 09:00 a 19:00
4,ACAHUALTEPEC,"Calle Colorín s/n, Colonia 1 , Santiago Acahua...",19.346043,-99.008134,De 09:00 a 19:00
...,...,...,...,...,...
299,YANCUIC,"PENDIENTEPENDIENTE 1 , LOS ANGELES",19.345913,-99.065821,De 09:00 a 19:00
300,YAUTLICA,"Av de las Torres S/N , Zona Rustica",19.333262,-98.983934,De 09:00 a 19:00
301,ZACATEPEC,"Calle Camino Real s/n 1 , Campestre Potrero",19.326209,-98.973121,De 09:00 a 19:00
302,ZENTLAPATL,"PUERTO ESCONDIDO 8 , ZENTLAPATL",19.355339,-99.309464,De 09:00 a 19:00


In [7]:
pilares_cdmx['sede'] = pilares_cdmx['nombre_pilar'].apply(limpiar_texto)
pilares_cdmx = pilares_cdmx[['sede','latitud','longitud']]
pilares_cdmx

,sede,latitud,longitud
0,100 metros,19.526523,-99.158897
1,16 de septiembre,19.280984,-99.137330
2,20 de noviembre,19.447458,-99.103176
3,8 de agosto,19.394845,-99.187943
4,acahualtepec,19.346043,-99.008134
...,...,...,...
299,yancuic,19.345913,-99.065821
300,yautlica,19.333262,-98.983934
301,zacatepec,19.326209,-98.973121
302,zentlapatl,19.355339,-99.309464


In [8]:
file_id = "1RHJsJ-sDzudqLBaPQwtUK31C1VuaV7ZGp0vyPzQkfus"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"

res = requests.get(url)
cve_pilares = pd.read_excel(BytesIO(res.content))
cve_pilares['sede'] = cve_pilares['nom_pilar_lista'].apply(limpiar_texto)
cve_pilares

,id_pilar,nombre_pilar,nom_pilar_lista,sede
0,069_NT_GAM,100 METROS,100 METROS,100 metros
1,006_SU_XOC,2025-09-16 00:00:00,16 DE SEPTIEMBRE,16 de septiembre
2,249_CO_VCA,2025-11-20 00:00:00,20 DE NOVIEMBRE,20 de noviembre
3,128_CO_BEJ,2025-08-08 00:00:00,8 DE AGOSTO,8 de agosto
4,011_IZ_IZP,ACAHUALTEPEC,ACAHUALTEPEC,acahualtepec
...,...,...,...,...
299,296_IZ_IZP,YANCUIC,YANCUIC,yancuic
300,298_IZ_IZT,YAUTLICA,YAUTLICA,yautlica
301,272_IZ_IZP,ZACATEPEC,ZACATEPEC,zacatepec
302,302_PT_CUM,ZENTLAPATL,ZENTLAPATL,zentlapatl


In [9]:
cve_pilares = pd.merge(cve_pilares, pilares_cdmx, on="sede", how="left")
cve_pilares
cve_pilares= cve_pilares[["id_pilar", "sede", "latitud", "longitud"]]

In [10]:
#Diccionario de zonas
map_pilars = {
    "CO": "Centro oriente",
    "IZ": "Iztapalapa",
    "NT": "Norte",
    "PT": "Poniente",
    "SP": "Sur poniente",
    "SU": "Sur"}

In [11]:
##Asignación de zonas de pilares
# Extraer el texto entre dos guiones que corresponden a la clave de la zona
cve_pilares["cve_zona"] = cve_pilares["id_pilar"].str.extract(r"_(.*?)_")

# Asignar la descripción según la clave
cve_pilares["pilar_zona"] = cve_pilares["cve_zona"].map(map_pilars)
cve_pilares = cve_pilares[["id_pilar", "cve_zona", "pilar_zona", "sede","latitud", "longitud"]]
cve_pilares

,id_pilar,cve_zona,pilar_zona,sede,latitud,longitud
0,069_NT_GAM,NT,Norte,100 metros,19.526523,-99.158897
1,006_SU_XOC,SU,Sur,16 de septiembre,19.280984,-99.137330
2,249_CO_VCA,CO,Centro oriente,20 de noviembre,19.447458,-99.103176
3,128_CO_BEJ,CO,Centro oriente,8 de agosto,19.394845,-99.187943
4,011_IZ_IZP,IZ,Iztapalapa,acahualtepec,19.346043,-99.008134
...,...,...,...,...,...,...
299,296_IZ_IZP,IZ,Iztapalapa,yancuic,19.345913,-99.065821
300,298_IZ_IZT,IZ,Iztapalapa,yautlica,19.333262,-98.983934
301,272_IZ_IZP,IZ,Iztapalapa,zacatepec,19.326209,-98.973121
302,302_PT_CUM,PT,Poniente,zentlapatl,19.355339,-99.309464


In [12]:
#Hacer capa de puntos los pilares (actividades)
geometry = [Point(xy) for xy in zip(cve_pilares['longitud'], cve_pilares['latitud'])]
cve_pilares_gdf = gpd.GeoDataFrame(cve_pilares, geometry=geometry, crs="EPSG:4326")
#cve_pilares_gdf = cve_pilares_gdf.drop_duplicates(subset=['geometry'])
cve_pilares_gdf

,id_pilar,cve_zona,pilar_zona,sede,latitud,longitud,geometry
0,069_NT_GAM,NT,Norte,100 metros,19.526523,-99.158897,POINT (-99.1589 19.52652)
1,006_SU_XOC,SU,Sur,16 de septiembre,19.280984,-99.137330,POINT (-99.13733 19.28098)
2,249_CO_VCA,CO,Centro oriente,20 de noviembre,19.447458,-99.103176,POINT (-99.10318 19.44746)
3,128_CO_BEJ,CO,Centro oriente,8 de agosto,19.394845,-99.187943,POINT (-99.18794 19.39484)
4,011_IZ_IZP,IZ,Iztapalapa,acahualtepec,19.346043,-99.008134,POINT (-99.00813 19.34604)
...,...,...,...,...,...,...,...
299,296_IZ_IZP,IZ,Iztapalapa,yancuic,19.345913,-99.065821,POINT (-99.06582 19.34591)
300,298_IZ_IZT,IZ,Iztapalapa,yautlica,19.333262,-98.983934,POINT (-98.98393 19.33326)
301,272_IZ_IZP,IZ,Iztapalapa,zacatepec,19.326209,-98.973121,POINT (-98.97312 19.32621)
302,302_PT_CUM,PT,Poniente,zentlapatl,19.355339,-99.309464,POINT (-99.30946 19.35534)


### Actividades de consulta

In [13]:
file_id = "1RwQfjjdz81S42a6msXhmiJGrRyrhQdis"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

res = requests.get(url)

In [14]:
eventos_consulta = pd.read_excel(BytesIO(res.content))
eventos_consulta = clean_names(eventos_consulta)
eventos_consulta['sede'] = eventos_consulta['sede'].apply(limpiar_texto)
eventos_consulta = eventos_consulta[eventos_consulta['mecanismo'].str.contains(r'^taller', case=False, na=False)]
eventos_consulta = eventos_consulta.merge(cve_pilares, on="sede", how="left")
eventos_consulta.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,consulta,mecanismo,sede,start_date,start_time,direccion,alcaldia,googlemaps,responsable_ipdp,apoyo_externo,observaciones,id_pilar,cve_zona,pilar_zona,latitud,longitud
0,Consulta Pública,"Taller participativo ""de los problemas a las p...",rini templeton,2025-11-11,10:00,"Calzada Real de San Martin No. 270, Col. Reyno...",Azcapotzalco,https://maps.app.goo.gl/Tq2KjrGYPCVELcaK9,Diana Pérez Cisneros y Erick Serna Luna,x,1a Sesión,192_NT_AZC,NT,Norte,19.497553,-99.187815
1,Consulta Pública,"Taller participativo ""de los problemas a las p...",ernesto che guevara,2025-11-11,10:00,"Carlos Pacheco S/N, Col. Ejército de Oriente Z...",Iztapalapa,https://maps.app.goo.gl/5aPTDweSY9VNbUJH6,Virginia Gizeh Piña Morales y Diego Ruiz Andoney,x,1a Sesión,070_IZ_IZP,IZ,Iztapalapa,NaN,NaN
2,Consulta Pública,"Taller participativo ""de los problemas a las p...",arbol del conocimiento,2025-11-11,10:00,"Luis G. Urbina S/N, Col. Lomas de Becerra",Álvaro Obregón,https://maps.app.goo.gl/62t3vnAHsyACgMux7,Jaime Vera Alpuche y Brenda Roxana Méndez Cadena,x,1a Sesión,043_PT_AOB,PT,Poniente,19.384272,-99.222057
3,Consulta Pública,"Taller participativo ""de los problemas a las p...",salvador allende,2025-11-11,10:00,"Ricardo Flores Magón S/N, Col. Olivar del Cond...",Álvaro Obregón,https://maps.app.goo.gl/u5MQbnVuEhjQDk3g6,Eduardo Rodríguez Parra,x,1a Sesión,NaN,NaN,NaN,NaN,NaN
4,Consulta Pública,"Taller participativo ""de los problemas a las p...",san jose,2025-11-11,11:00,"Agustín Lara S/N, Col. San José",Tláhuac,https://maps.app.goo.gl/bUSCMEJajv6YPmT67,Israel Rodríguez Ramírez y Andrea Mancera Flores,x,1a Sesión,275_SU_TLH,SU,Sur,19.276049,-98.993618


In [15]:
valores = [
    'Taller participativo "de los problemas a las propuestas"',
    'Taller participativo "la Ciudad que queremos"'
]

talleres_pilares_prog = eventos_consulta[eventos_consulta['mecanismo'].isin(valores)]

#Eliminar talleres pospuestos
#talleres_pilares_prog= talleres_pilares_prog[talleres_pilares_prog["responsable_ipdp"] != "Pospuesto"]

talleres_pilares_prog["mecanismo"] = np.where(
    talleres_pilares_prog["mecanismo"].str.contains("de los problemas", case=False, na=False),
    "1a sesión",
    "2da sesión"
)

talleres_pilares_prog["sede"]= talleres_pilares_prog["sede"].str.replace("PILARES", "")
talleres_pilares_prog["sede"]=talleres_pilares_prog["sede"].str.upper()
#Eliminar espacios adelante y atrás
talleres_pilares_prog["sede"] = talleres_pilares_prog["sede"].str.strip()

#Total de talleres
print("Total de talleres:", len(talleres_pilares_prog))
print("La fecha inicial de talleres fue", min(talleres_pilares_prog["start_date"]))
print("La fecha final de talleres fue", max(talleres_pilares_prog["start_date"]))

talleres_pilares_prog.head()

/tmp/ipython-input-360631192.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  talleres_pilares_prog["mecanismo"] = np.where(
/tmp/ipython-input-360631192.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  talleres_pilares_prog["sede"]= talleres_pilares_prog["sede"].str.replace("PILARES", "")
/tmp/ipython-input-360631192.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docum

Total de talleres: 460
La fecha inicial de talleres fue 2025-11-11 00:00:00
La fecha final de talleres fue 2025-12-06 00:00:00


,consulta,mecanismo,sede,start_date,start_time,direccion,alcaldia,googlemaps,responsable_ipdp,apoyo_externo,observaciones,id_pilar,cve_zona,pilar_zona,latitud,longitud
0,Consulta Pública,1a sesión,RINI TEMPLETON,2025-11-11,10:00,"Calzada Real de San Martin No. 270, Col. Reyno...",Azcapotzalco,https://maps.app.goo.gl/Tq2KjrGYPCVELcaK9,Diana Pérez Cisneros y Erick Serna Luna,x,1a Sesión,192_NT_AZC,NT,Norte,19.497553,-99.187815
1,Consulta Pública,1a sesión,ERNESTO CHE GUEVARA,2025-11-11,10:00,"Carlos Pacheco S/N, Col. Ejército de Oriente Z...",Iztapalapa,https://maps.app.goo.gl/5aPTDweSY9VNbUJH6,Virginia Gizeh Piña Morales y Diego Ruiz Andoney,x,1a Sesión,070_IZ_IZP,IZ,Iztapalapa,NaN,NaN
2,Consulta Pública,1a sesión,ARBOL DEL CONOCIMIENTO,2025-11-11,10:00,"Luis G. Urbina S/N, Col. Lomas de Becerra",Álvaro Obregón,https://maps.app.goo.gl/62t3vnAHsyACgMux7,Jaime Vera Alpuche y Brenda Roxana Méndez Cadena,x,1a Sesión,043_PT_AOB,PT,Poniente,19.384272,-99.222057
3,Consulta Pública,1a sesión,SALVADOR ALLENDE,2025-11-11,10:00,"Ricardo Flores Magón S/N, Col. Olivar del Cond...",Álvaro Obregón,https://maps.app.goo.gl/u5MQbnVuEhjQDk3g6,Eduardo Rodríguez Parra,x,1a Sesión,NaN,NaN,NaN,NaN,NaN
4,Consulta Pública,1a sesión,SAN JOSE,2025-11-11,11:00,"Agustín Lara S/N, Col. San José",Tláhuac,https://maps.app.goo.gl/bUSCMEJajv6YPmT67,Israel Rodríguez Ramírez y Andrea Mancera Flores,x,1a Sesión,275_SU_TLH,SU,Sur,19.276049,-98.993618


### Capa de límite de alcaldías

In [16]:

drive.mount('/content/drive')


Mounted at /content/drive


In [17]:
ruta = "/content/drive/MyDrive/SHP archivos vectoriales/09mun.shp"
alcaldias = gpd.read_file(ruta).to_crs(epsg=4326)
alcaldias = clean_names(alcaldias)
alcaldias = alcaldias[['nomgeo','geometry']]
alcaldias

,nomgeo,geometry
0,Tlalpan,"POLYGON ((-99.21985 19.31212, -99.21966 19.311..."
1,Milpa Alta,"POLYGON ((-98.9988 19.22769, -98.99815 19.2275..."
2,Cuauhtémoc,"POLYGON ((-99.14174 19.46582, -99.14165 19.465..."
3,Iztapalapa,"POLYGON ((-99.05813 19.40072, -99.05813 19.400..."
4,La Magdalena Contreras,"POLYGON ((-99.20785 19.33787, -99.20752 19.337..."
5,Álvaro Obregón,"POLYGON ((-99.1922 19.40372, -99.19219 19.4032..."
6,Coyoacán,"POLYGON ((-99.17169 19.35956, -99.17083 19.359..."
7,Miguel Hidalgo,"POLYGON ((-99.20084 19.47335, -99.20068 19.473..."
8,Cuajimalpa de Morelos,"POLYGON ((-99.25771 19.40474, -99.25772 19.404..."
9,Iztacalco,"POLYGON ((-99.05591 19.42219, -99.05579 19.422..."


In [18]:
#Alargar la url de google maps
eventos_consulta['url_larga'] = eventos_consulta['googlemaps'].apply(expandir_url)
#Extraer las coordenadas
eventos_consulta[['latitud','longitud']] = eventos_consulta['url_larga'] \
    .apply(lambda x: pd.Series(extraer_coordenadas(x)))
eventos_consulta.head()


,consulta,mecanismo,sede,start_date,start_time,direccion,alcaldia,googlemaps,responsable_ipdp,apoyo_externo,observaciones,id_pilar,cve_zona,pilar_zona,latitud,longitud,url_larga
0,Consulta Pública,"Taller participativo ""de los problemas a las p...",rini templeton,2025-11-11,10:00,"Calzada Real de San Martin No. 270, Col. Reyno...",Azcapotzalco,https://maps.app.goo.gl/Tq2KjrGYPCVELcaK9,Diana Pérez Cisneros y Erick Serna Luna,x,1a Sesión,192_NT_AZC,NT,Norte,19.497389,-99.187471,https://www.google.com.mx/maps/place/PILARES+R...
1,Consulta Pública,"Taller participativo ""de los problemas a las p...",ernesto che guevara,2025-11-11,10:00,"Carlos Pacheco S/N, Col. Ejército de Oriente Z...",Iztapalapa,https://maps.app.goo.gl/5aPTDweSY9VNbUJH6,Virginia Gizeh Piña Morales y Diego Ruiz Andoney,x,1a Sesión,070_IZ_IZP,IZ,Iztapalapa,19.366623,-99.030093,https://www.google.com.mx/maps/place/Pilares+C...
2,Consulta Pública,"Taller participativo ""de los problemas a las p...",arbol del conocimiento,2025-11-11,10:00,"Luis G. Urbina S/N, Col. Lomas de Becerra",Álvaro Obregón,https://maps.app.goo.gl/62t3vnAHsyACgMux7,Jaime Vera Alpuche y Brenda Roxana Méndez Cadena,x,1a Sesión,043_PT_AOB,PT,Poniente,19.384251,-99.221935,https://www.google.com/maps/place/PILARES+%C3%...
3,Consulta Pública,"Taller participativo ""de los problemas a las p...",salvador allende,2025-11-11,10:00,"Ricardo Flores Magón S/N, Col. Olivar del Cond...",Álvaro Obregón,https://maps.app.goo.gl/u5MQbnVuEhjQDk3g6,Eduardo Rodríguez Parra,x,1a Sesión,NaN,NaN,NaN,19.374773,-99.278864,https://www.google.com.mx/maps/place/PILARES+L...
4,Consulta Pública,"Taller participativo ""de los problemas a las p...",san jose,2025-11-11,11:00,"Agustín Lara S/N, Col. San José",Tláhuac,https://maps.app.goo.gl/bUSCMEJajv6YPmT67,Israel Rodríguez Ramírez y Andrea Mancera Flores,x,1a Sesión,275_SU_TLH,SU,Sur,19.276095,-98.996499,https://www.google.com.mx/maps/place/PILARES+S...


In [19]:
#Hacer capa de puntos los pilares (actividades)
geometry = [Point(xy) for xy in zip(eventos_consulta['longitud'], eventos_consulta['latitud'])]
eventos_consulta_gdf = gpd.GeoDataFrame(eventos_consulta, geometry=geometry, crs="EPSG:4326")
eventos_consulta_gdf = eventos_consulta_gdf.drop_duplicates(subset=['geometry'])
eventos_consulta_gdf = eventos_consulta_gdf[["id_pilar","cve_zona",	"pilar_zona","sede", "geometry"]]
eventos_consulta_gdf

,id_pilar,cve_zona,pilar_zona,sede,geometry
0,192_NT_AZC,NT,Norte,rini templeton,POINT (-99.18747 19.49739)
1,070_IZ_IZP,IZ,Iztapalapa,ernesto che guevara,POINT (-99.03009 19.36662)
2,043_PT_AOB,PT,Poniente,arbol del conocimiento,POINT (-99.22193 19.38425)
3,NaN,NaN,NaN,salvador allende,POINT (-99.27886 19.37477)
4,275_SU_TLH,SU,Sur,san jose,POINT (-98.9965 19.2761)
...,...,...,...,...,...
237,120_CO_BEJ,CO,Centro oriente,carlos monsivais,POINT (-99.1455 19.36423)
268,150_SP_TLP,SP,Sur poniente,topilejo,POINT (-99.14409 19.19872)
361,010_CO_IZT,CO,Centro oriente,agricola pantitlan,POINT (-99.0709 19.41542)
478,NaN,NaN,NaN,centro de estudios interculturales nezahualcoyotl,POINT (NaN NaN)


In [20]:
print(alcaldias.crs)
print(cve_pilares_gdf.crs)
print(eventos_consulta_gdf.crs)


EPSG:4326
EPSG:4326
EPSG:4326


In [21]:
#pilares_con_alcaldia = cve_pilares_gdf.sjoin(alcaldias, how="left")
#pilares_con_alcaldia
#pilares_con_alcaldia.to_file("pilares_con_alcaldia.geojson", driver="GeoJSON")
#!ls /content
#from google.colab import files
#files.download("pilares_con_alcaldia.geojson")

,id_pilar,cve_zona,pilar_zona,sede,latitud,longitud,geometry,index_right,nomgeo
0,069_NT_GAM,NT,Norte,100 metros,19.526523,-99.158897,POINT (-99.1589 19.52652),12.0,Gustavo A. Madero
1,006_SU_XOC,SU,Sur,16 de septiembre,19.280984,-99.137330,POINT (-99.13733 19.28098),11.0,Xochimilco
2,249_CO_VCA,CO,Centro oriente,20 de noviembre,19.447458,-99.103176,POINT (-99.10318 19.44746),14.0,Venustiano Carranza
3,128_CO_BEJ,CO,Centro oriente,8 de agosto,19.394845,-99.187943,POINT (-99.18794 19.39484),15.0,Benito Juárez
4,011_IZ_IZP,IZ,Iztapalapa,acahualtepec,19.346043,-99.008134,POINT (-99.00813 19.34604),3.0,Iztapalapa
...,...,...,...,...,...,...,...,...,...
299,296_IZ_IZP,IZ,Iztapalapa,yancuic,19.345913,-99.065821,POINT (-99.06582 19.34591),3.0,Iztapalapa
300,298_IZ_IZT,IZ,Iztapalapa,yautlica,19.333262,-98.983934,POINT (-98.98393 19.33326),3.0,Iztapalapa
301,272_IZ_IZP,IZ,Iztapalapa,zacatepec,19.326209,-98.973121,POINT (-98.97312 19.32621),3.0,Iztapalapa
302,302_PT_CUM,PT,Poniente,zentlapatl,19.355339,-99.309464,POINT (-99.30946 19.35534),8.0,Cuajimalpa de Morelos


In [26]:
# Unión espacial
pilares_con_alcaldia = eventos_consulta_gdf.sjoin(alcaldias, how="left")

In [27]:
pilares_con_alcaldia = pd.DataFrame(pilares_con_alcaldia)
pilares_con_alcaldia = pilares_con_alcaldia[["id_pilar","cve_zona",	"pilar_zona","sede", "nomgeo"]]
pilares_con_alcaldia

,id_pilar,cve_zona,pilar_zona,sede,nomgeo
0,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco
1,070_IZ_IZP,IZ,Iztapalapa,ernesto che guevara,Iztapalapa
2,043_PT_AOB,PT,Poniente,arbol del conocimiento,Álvaro Obregón
3,NaN,NaN,NaN,salvador allende,Cuajimalpa de Morelos
4,275_SU_TLH,SU,Sur,san jose,Tláhuac
...,...,...,...,...,...
237,120_CO_BEJ,CO,Centro oriente,carlos monsivais,Benito Juárez
268,150_SP_TLP,SP,Sur poniente,topilejo,Tlalpan
361,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco
478,NaN,NaN,NaN,centro de estudios interculturales nezahualcoyotl,NaN


In [28]:
#Agregar tabla de textos de problematicas para analizarlos por zona
file_id = "1gEzHMJA9xa8JW8ksedTBDfcMZIOXFqYo2E9b7rNySbk"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"

res = requests.get(url)
textos_talleres = pd.read_excel(BytesIO(res.content))
textos_talleres.head()


,id_pilar,id_sesion,id_texto,fecha,sub_tipo_evento,lugar_taller,ejecuta_taller,texto,tema,tipo_texto,linea_transformacion,texto_clean
0,281_SU_XOC,1,281_SU_XOC_001_098,2025-11-13,"1a sesión: ""De los problemas a las propuestas""",VIVERO NEZAHUALCÓYOTL,Sí,"Escasez del agua, el agua está contaminada","Medio ambiente, agua y riesgos",problematica,NaN,escasez agua agua contaminada
1,281_SU_XOC,1,281_SU_XOC_001_052,2025-11-13,"1a sesión: ""De los problemas a las propuestas""",VIVERO NEZAHUALCÓYOTL,Sí,"hay residuos tóxicos, basura","Medio ambiente, agua y riesgos",problematica,NaN,residuos toxicos basura
2,281_SU_XOC,1,281_SU_XOC_001_070,2025-11-13,"1a sesión: ""De los problemas a las propuestas""",VIVERO NEZAHUALCÓYOTL,Sí,Hay pérdida de áreas verdes,"Medio ambiente, agua y riesgos",problematica,NaN,perdida areas verdes
3,281_SU_XOC,1,281_SU_XOC_001_090,2025-11-13,"1a sesión: ""De los problemas a las propuestas""",VIVERO NEZAHUALCÓYOTL,Sí,hay colapso en el drenaje,"Medio ambiente, agua y riesgos",problematica,NaN,colapso drenaje
4,281_SU_XOC,1,281_SU_XOC_001_014,2025-11-13,"1a sesión: ""De los problemas a las propuestas""",VIVERO NEZAHUALCÓYOTL,Sí,existe sobrepoblación de animales callejeros,"Medio ambiente, agua y riesgos",problematica,NaN,existe sobrepoblacion animales callejeros


In [29]:
textos_talleres = pd.merge(pilares_con_alcaldia, textos_talleres, on="id_pilar", how="left")
textos_talleres = textos_talleres[textos_talleres["id_pilar"].notna()]
textos_talleres

,id_pilar,cve_zona,pilar_zona,sede,nomgeo,id_sesion,id_texto,fecha,sub_tipo_evento,lugar_taller,ejecuta_taller,texto,tema,tipo_texto,linea_transformacion,texto_clean
0,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_022,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Coladeras tapaderas,"Medio ambiente, agua y riesgos",problematica,NaN,coladeras tapaderas
1,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_035,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Agua potable con tierra,"Medio ambiente, agua y riesgos",problematica,NaN,agua potable tierra
2,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_055,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Basura en la calle,"Medio ambiente, agua y riesgos",problematica,NaN,basura calle
3,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_052,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Falta de áreas verdes,"Medio ambiente, agua y riesgos",problematica,NaN,falta areas verdes
4,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_038,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Smog,"Medio ambiente, agua y riesgos",problematica,NaN,smog
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19948,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_057,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Jornada laboral de 40 horas:Fortalecer proyect...,"Cultura, economía, educación, ciencia y trabajo",solucion,NaN,jornada laboral horasfortalecer proyectos comu...
19949,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_052,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Educación gratuita sobre economía para el hoga...,"Cultura, economía, educación, ciencia y trabajo",solucion,NaN,educacion gratuita economia hogar trabajo
19950,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_055,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Designación y protección de espacios agrícolas,"Cultura, economía, educación, ciencia y trabajo",solucion,NaN,designacion proteccion espacios agricolas
19951,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_005,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Mayor consumo local,"Cultura, economía, educación, ciencia y trabajo",solucion,NaN,mayor consumo local


Problematicas agrupadas por zona y por alcaldia

In [31]:
df_prob = textos_talleres[textos_talleres["tipo_texto"] == "problematica"]
df = df_prob.copy()

df["texto_clean"] = ( df["texto_clean"]
    .fillna("")
    .astype(str)
    .str.strip())

# quitar strings basura
df.loc[df["texto_clean"].isin(["nan", "None", ""]), "texto_clean"] = ""

df

,id_pilar,cve_zona,pilar_zona,sede,nomgeo,id_sesion,id_texto,fecha,sub_tipo_evento,lugar_taller,ejecuta_taller,texto,tema,tipo_texto,linea_transformacion,texto_clean
0,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_022,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Coladeras tapaderas,"Medio ambiente, agua y riesgos",problematica,NaN,coladeras tapaderas
1,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_035,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Agua potable con tierra,"Medio ambiente, agua y riesgos",problematica,NaN,agua potable tierra
2,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_055,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Basura en la calle,"Medio ambiente, agua y riesgos",problematica,NaN,basura calle
3,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_052,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Falta de áreas verdes,"Medio ambiente, agua y riesgos",problematica,NaN,falta areas verdes
4,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_038,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Smog,"Medio ambiente, agua y riesgos",problematica,NaN,smog
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19915,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_015,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Burocracia para obtener un trabajo,"Cultura, economía, educación, ciencia y trabajo",problematica,NaN,burocracia obtener trabajo
19916,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_038,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,"Trabajos en cultura, educación y trabajo mal p...","Cultura, economía, educación, ciencia y trabajo",problematica,NaN,trabajos cultura educacion trabajo mal pagados
19917,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_014,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Horas extensas y mal pagadas,"Cultura, economía, educación, ciencia y trabajo",problematica,NaN,horas extensas mal pagadas
19918,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_042,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Poca difusión de talleres científicos y cultur...,"Cultura, economía, educación, ciencia y trabajo",problematica,NaN,poca difusion talleres cientificos culturales


In [46]:
# Modelo de embeddings
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def clustering_por_grupo(sub):
    textos = sub["texto_clean"].tolist()

    if len(textos) < 5:
        sub["cluster_id"] = -1
        sub["embedding"] = [None] * len(sub)
        return sub

    # Embeddings
    emb = normalize(model.encode(textos, show_progress_bar=False))

    # HDBSCAN
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=2,
        metric="euclidean",
        cluster_selection_method="leaf"
    )

    sub["cluster_id"] = clusterer.fit_predict(emb)
    sub["embedding"] = list(emb)

    return sub

In [ ]:
def nombrar_clusters_por_nomgeo(df):
    nombres = {}

    for (nomgeo, cid), sub in df.groupby(["nomgeo", "cluster_id"]):
        if cid == -1:
            nombres[(nomgeo, cid)] = "Sin clasificar"
            continue

        emb = np.vstack(sub["embedding"].values)
        centro = emb.mean(axis=0).reshape(1, -1)

        sims = cosine_similarity(emb, centro).flatten()
        idx = sims.argmax()

        nombres[(nomgeo, cid)] = sub.iloc[idx]["texto_clean"]

    df["cluster_nombre"] = df.set_index(["nomgeo", "cluster_id"]).index.map(nombres)
    return df

In [48]:
# Aplicar clustering por cada grupo de 'nomgeo'
df2 = (df.groupby("nomgeo", group_keys=False).apply(clustering_por_grupo).reset_index(drop=True))
df2 = nombrar_clusters_por_nomgeo(df2)
df2

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

In [53]:
#Contabilización de las categorias por alcaldía
problemas_por_alcaldia = (df2.loc[df2["cluster_id"] != -1].groupby(["nomgeo", "cluster_nombre"]).size().reset_index(name="n"))
top2_por_alcaldia = (problemas_por_alcaldia.sort_values(["nomgeo", "n"], ascending=[True, False]).groupby("nomgeo").head(2).reset_index(drop=True))
top2_por_alcaldia

,nomgeo,cluster_nombre,n
0,Azcapotzalco,ambulantaje,5
1,Azcapotzalco,corrupcion elementos policiacos,8
2,Azcapotzalco,desabasto medicinas centros salud,27
3,Azcapotzalco,falta aprendizaje escuelas,8
4,Azcapotzalco,falta areas verdes,6
...,...,...,...
232,Álvaro Obregón,invasion banquetas motos estacionadas carros n...,8
233,Álvaro Obregón,privacion vivienda accesible,18
234,Álvaro Obregón,suficientes espacios culturales,8
235,Álvaro Obregón,transporte publico deficiente,20


In [57]:
# Aplicar clustering por cada grupo de 'zona_id'
df3= (df.groupby("pilar_zona", group_keys=False).apply(clustering_por_grupo).reset_index(drop=True))
df3 = nombrar_clusters_por_nomgeo(df3)
df3

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12

,id_pilar,cve_zona,pilar_zona,sede,nomgeo,id_sesion,id_texto,fecha,sub_tipo_evento,lugar_taller,ejecuta_taller,texto,tema,tipo_texto,linea_transformacion,texto_clean,cluster_id,embedding,cluster_nombre
0,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_022,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Coladeras tapaderas,"Medio ambiente, agua y riesgos",problematica,NaN,coladeras tapaderas,-1,"[0.022704152, -0.0022501927, -0.0058757784, 0....",Sin clasificar
1,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_035,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Agua potable con tierra,"Medio ambiente, agua y riesgos",problematica,NaN,agua potable tierra,-1,"[0.031534567, 0.017546441, 0.061602104, -0.003...",Sin clasificar
2,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_055,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Basura en la calle,"Medio ambiente, agua y riesgos",problematica,NaN,basura calle,48,"[0.060834598, -0.0022950822, 0.015904356, -0.0...",basura calles
3,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_052,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Falta de áreas verdes,"Medio ambiente, agua y riesgos",problematica,NaN,falta areas verdes,33,"[0.16075781, -0.0029470783, 0.059309594, 0.054...",falta areas verdes
4,192_NT_AZC,NT,Norte,rini templeton,Azcapotzalco,1.0,192_NT_AZC_001_038,2025-11-11,"1a sesión: ""De los problemas a las propuestas""",RINI TEMPLETON,Sí,Smog,"Medio ambiente, agua y riesgos",problematica,NaN,smog,-1,"[0.003216679, 0.038608253, -0.00693269, 0.0647...",Sin clasificar
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6124,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_015,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Burocracia para obtener un trabajo,"Cultura, economía, educación, ciencia y trabajo",problematica,NaN,burocracia obtener trabajo,-1,"[0.021148069, 0.0053769224, -0.02589062, -0.06...",Sin clasificar
6125,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_038,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,"Trabajos en cultura, educación y trabajo mal p...","Cultura, economía, educación, ciencia y trabajo",problematica,NaN,trabajos cultura educacion trabajo mal pagados,-1,"[0.025662877, 0.054221842, 0.01878704, 0.05928...",Sin clasificar
6126,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_014,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Horas extensas y mal pagadas,"Cultura, economía, educación, ciencia y trabajo",problematica,NaN,horas extensas mal pagadas,-1,"[-0.018690335, -0.008259663, 0.0135649275, 0.0...",Sin clasificar
6127,010_CO_IZT,CO,Centro oriente,agricola pantitlan,Iztacalco,1.0,010_CO_IZT_001_042,2025-12-02,"1a sesión: ""De los problemas a las propuestas""",AGRÍCOLA PANTITLÁN,Sí,Poca difusión de talleres científicos y cultur...,"Cultura, economía, educación, ciencia y trabajo",problematica,NaN,poca difusion talleres cientificos culturales,-1,"[0.07205165, 0.07968952, 0.018425932, 0.023758...",Sin clasificar


In [58]:
#Contabilización de las categorias por zona
problemas_por_zona = (df3.loc[df3["cluster_id"] != -1].groupby(["pilar_zona", "cluster_nombre"]).size().reset_index(name="n"))
top2_por_zona = (problemas_por_zona.sort_values(["pilar_zona", "n"], ascending=[True, False]).groupby("pilar_zona").head(2).reset_index(drop=True))
top2_por_zona

,pilar_zona,cluster_nombre,n
0,Centro oriente,desabasto medicinas faltan hospitales especial...,14
1,Centro oriente,basura calles,12
2,Iztapalapa,baches,33
3,Iztapalapa,baches calles,25
4,Norte,falta vivienda,24
5,Norte,falta luminaria espacios publicos,22
6,Poniente,baches,18
7,Poniente,insuficientes clinicas salud falta medicamentos,18
8,Sur,falta medicamentos centro salud,22
9,Sur,poda arboles,17
